# الفصل 27: التقليم وضغط النماذج: Pruning

> الأبحاث تُظهر مرارًا أن نماذج اللغة الكبيرة مُفرِطة المعاملات (Over-parameterized) إلى حد كبير — نسبة كبيرة من الأوزان مساهمتها في الأداء النهائي ضئيلة جدًا أو شبه معدومة.

- **الجزء:** 6 — إعادة استخدام النماذج
- **الفصل:** 27
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 26: دمج النماذج: Model Merging
- **الوسوم:** Pruning، SparseGPT، Wanda، LLM-Pruner

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch27-pruning.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-06-model-reuse%2Fch27-pruning.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-06-model-reuse/ch27-pruning.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch27-pruning.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch27-pruning.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch27-pruning.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** peft, torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **خلايا ثقيلة:** قد تستغرق خلايا التدريب/التحميل وقتًا طويلًا. شغّلها خلية بخية، وقلّل `batch_size` أو استخدم نموذجًا أصغر عند الحاجة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("peft", "peft"),
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch27-pruning.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-06-model-reuse%2Fch27-pruning.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-06-model-reuse/ch27-pruning.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch27-pruning.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch27-pruning.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch27-pruning.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> الأبحاث تُظهر مرارًا أن نماذج اللغة الكبيرة مُفرِطة المعاملات (Over-parameterized) إلى حد كبير — نسبة كبيرة من الأوزان مساهمتها في الأداء النهائي ضئيلة جدًا أو شبه معدومة.

> **بيانات الفصل**
>
> - **الجزء:** 6 — إعادة استخدام النماذج
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 26: دمج النماذج: Model Merging


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم التقليم غير البنيوي مقابل البنيوي
- فهم Magnitude Pruning: البداية البسيطة
- فهم SparseGPT: التقليم المُدرَك للخطأ (Error-Aware)


## 1. لماذا يهمّك هذا الموضوع؟

🟢 طبقة الحدس
الأبحاث تُظهر مرارًا أن نماذج اللغة الكبيرة مُفرِطة المعاملات (Over-parameterized) إلى حد كبير — نسبة كبيرة من الأوزان مساهمتها في الأداء النهائي ضئيلة جدًا أو شبه معدومة. Pruning هي عملية "اكتشاف وإزالة" هذه الأوزان الزائدة، بهدف تقليل حجم النموذج وتسريع الاستدلال دون المساس الجوهري بالأداء — فكرة أقرب لـ"نحت تمثال من كتلة رخام" أكثر من "بناء من الصفر".


## 2. الخريطة الذهنية


```text
التقليم (Pruning)
│
├── 1) غير بنيوي (Unstructured)
│   ├── Magnitude Pruning        |W| فقط، بسيط وسريع
│   ├── SparseGPT                Hessian + تحديث تعويضي (OBS)
│   └── Wanda                    |W| × ||X||، بسيط وفعّال
│
└── 2) بنيوي (Structured)
    ├── إزالة رؤوس انتباه / قنوات FFN
    ├── إزالة طبقات كاملة
    └── Recovery Fine-tuning (LoRA) لإعادة الأداء
```


السؤال الحاسم قبل اختيار التقنية: **هل تريد توفير ذاكرة فقط، أم تسريعًا فعليًا
على العتاد؟** التقليم غير البنيوي يوفّر ذاكرة نظرية كبيرة لكنه لا يُسرّع
المصفوفات الكثيفة على GPU عادي. التقليم البنيوي يُنتج مصفوفات أصغر فعلًا،
لكنه محدود بنسبة تقليم أقل ويحتاج مرحلة تعافٍ.


## 3. الشرح المتدرج

### 3.1 التقليم غير البنيوي مقابل البنيوي

### 3.2 Magnitude Pruning: البداية البسيطة


In [ ]:
import torch
import torch.nn as nn

def magnitude_pruning_unstructured(model: nn.Module, sparsity_ratio: float = 0.5):
    """
    أبسط طريقة تقليم: إزالة الأوزان ذات أصغر قيمة مطلقة (Magnitude)
    بافتراض أنها الأقل مساهمة في المخرج النهائي. فعّالة كخط أساس
    لكنها "عمياء" عن السياق الفعلي لاستخدام الوزن أثناء الاستدلال.
    """
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            weight = module.weight.data
            flat_weight = weight.abs().flatten()
            k = int(len(flat_weight) * sparsity_ratio)
            if k == 0:
                continue
            threshold = torch.topk(flat_weight, k, largest=False).values.max()
            mask = weight.abs() > threshold
            module.weight.data = weight * mask

            print(f"{name}: نسبة التقليم الفعلية = "
                  f"{(mask == 0).float().mean().item():.2%}")
    return model


### 3.3 SparseGPT: التقليم المُدرَك للخطأ (Error-Aware)

🔴 طبقة التعمّق

المشكلة في Magnitude Pruning: إزالة وزن صغير القيمة قد لا تزال تُحدِث خطأً كبيرًا إذا كان مُدخَله (activation) كبيرًا جدًا. SparseGPT (Frantar & Alistarh, 2023) يحل هذا بحساب التقليم الأمثل الذي يُقلِّل خطأ إعادة البناء (Reconstruction Error) طبقة بطبقة، باستخدام تقنيات من الجبر الخطي (مصفوفة Hessian التقريبية).


In [ ]:
class SparseGPTPruner:
    """
    تطبيق مُبسَّط لمبدأ SparseGPT: بدل التقليم "الأعمى" بناءً على
    قيمة الوزن فقط، نحسب تقليمًا يُقلِّل خطأ إعادة بناء مخرجات
    الطبقة (باستخدام معلومات إحصائية عن توزيع المُدخلات الفعلية -
    مصفوفة Hessian التقريبية H = X^T X)، مع تحديث الأوزان المتبقية
    للتعويض عن إزالة الأوزان المُقلَّمة (مبدأ OBS - Optimal Brain Surgeon).
    """
    def __init__(self, layer: nn.Linear, sparsity_ratio: float = 0.5,
                 damp_percent: float = 0.01):
        self.layer = layer
        self.sparsity_ratio = sparsity_ratio
        self.damp_percent = damp_percent  # لتفادي عدم استقرار معكوس المصفوفة
        self.hessian_accumulator = torch.zeros(
            layer.in_features, layer.in_features
        )
        self.n_samples = 0

    def accumulate_hessian_statistics(self, layer_input: torch.Tensor):
        """
        تجميع إحصائيات H = X^T X عبر دفعات متعددة من بيانات معايرة
        (Calibration Data - عادة بضع مئات من العيّنات التمثيلية).
        """
        # layer_input: [batch * seq_len, in_features]
        batch_samples = layer_input.shape[0]
        self.hessian_accumulator *= self.n_samples / (self.n_samples + batch_samples)
        self.n_samples += batch_samples
        scaled_input = layer_input * (2.0 / self.n_samples) ** 0.5
        self.hessian_accumulator += scaled_input.T @ scaled_input

    def prune_with_weight_update(self) -> nn.Linear:
        """
        تطبيق التقليم مع تحديث تعويضي للأوزان المتبقية (مبدأ OBS
        المُبسَّط): لكل وزن نُزيله، نُحدِّث الأوزان الأخرى في نفس
        الصف لتعويض الخطأ الناتج قدر الإمكان، باستخدام معكوس
        مصفوفة Hessian.
        """
        H = self.hessian_accumulator.clone()
        damp = self.damp_percent * torch.mean(torch.diag(H))
        H += torch.eye(H.shape[0]) * damp  # تثبيت رقمي (damping)

        H_inv = torch.linalg.inv(H)
        weight = self.layer.weight.data.clone()  # [out_features, in_features]

        # درجة أهمية كل وزن = (قيمة الوزن)² / (قيمة H_inv القطرية المقابلة)
        # - هذا هو معيار OBS الكلاسيكي لتقدير "ثمن" إزالة كل وزن
        diag_H_inv = torch.diag(H_inv)
        importance_scores = (weight ** 2) / diag_H_inv.unsqueeze(0)

        n_to_prune = int(weight.shape[1] * self.sparsity_ratio)
        for row in range(weight.shape[0]):
            row_importance = importance_scores[row]
            prune_indices = torch.topk(
                row_importance, n_to_prune, largest=False
            ).indices

            for idx in prune_indices:
                # تحديث تعويضي لباقي الأوزان في نفس الصف (مبدأ OBS)
                delta = (weight[row, idx] / diag_H_inv[idx]) * H_inv[idx, :]
                weight[row, :] -= delta
                weight[row, idx] = 0.0  # التأكد من التصفير التام

        self.layer.weight.data = weight
        return self.layer


### 3.4 Wanda: بساطة فعّالة بدون تحديث الأوزان

🟡 طبقة الآلية

Wanda (Weights AND Activations) تحقق أداءً مقاربًا لـ SparseGPT لكن بدون أي حاجة لحساب معكوس مصفوفة Hessian المكلف — فقط بضرب قيمة الوزن في معيار (norm) التنشيط المقابل له.


In [ ]:
def wanda_pruning(layer: nn.Linear, calibration_activations: torch.Tensor,
                   sparsity_ratio: float = 0.5) -> nn.Linear:
    """
    Wanda (Sun et al., 2023): معيار تقليم بسيط جدًا لكنه فعّال بشكل
    مُدهش: درجة أهمية الوزن W_ij = |W_ij| × ||X_j||₂
    (قيمة الوزن المطلقة × معيار L2 لتنشيطات العمود المقابل عبر
    عيّنات المعايرة). لا حاجة لحساب Hessian أو تحديث تعويضي معقّد -
    فقط ترتيب وتصفير مباشر، مع نتائج منافسة جدًا لـ SparseGPT الأكثر تعقيدًا.
    """
    weight = layer.weight.data  # [out_features, in_features]

    # معيار L2 لكل عمود (ميزة إدخال) عبر عيّنات المعايرة
    activation_norms = calibration_activations.norm(p=2, dim=0)  # [in_features]

    # درجة الأهمية = |الوزن| × معيار تنشيط عموده
    importance_scores = weight.abs() * activation_norms.unsqueeze(0)

    n_to_prune_per_row = int(weight.shape[1] * sparsity_ratio)
    for row in range(weight.shape[0]):
        prune_indices = torch.topk(
            importance_scores[row], n_to_prune_per_row, largest=False
        ).indices
        weight[row, prune_indices] = 0.0

    layer.weight.data = weight
    return layer


⚖️ جدول مقارنة: Magnitude Pruning مقابل SparseGPT مقابل Wanda

| المعيار | Magnitude Pruning | SparseGPT | Wanda |
|---|---|---|---|
| يأخذ التنشيطات بالحسبان؟ | لا | نعم (عبر Hessian) | نعم (عبر معيار L2 بسيط) |
| يُحدِّث الأوزان المتبقية؟ | لا | نعم (تعويض OBS) | لا |
| التكلفة الحسابية | منخفضة جدًا | عالية (معكوس مصفوفة) | منخفضة جدًا |
| الأداء عند نسب تقليم عالية (>50%) | ضعيف | ممتاز | ممتاز (مُقارب لـ SparseGPT) |
| سهولة التطبيق | بسيطة جدًا | معقّدة نسبيًا | بسيطة جدًا |

### 3.5 LLM-Pruner: التقليم البنيوي الفعلي وإعادة التدريب

🟡 طبقة الآلية

للحصول على تسريع فعلي على أي هاردوير (وليس فقط توفير ذاكرة نظري)، نحتاج تقليمًا بنيويًا يُزيل مكوّنات كاملة.


In [ ]:
class StructuredLLMPruner:
    """
    تطبيق مفاهيمي لمبدأ LLM-Pruner (Ma et al., 2023): تحديد "مجموعات
    مترابطة" من الأوزان (Coupled Structures - مثل رأس انتباه كامل أو
    قناة FFN كاملة بكل ما يتصل بها من أوزان دخل وخرج)، تقدير أهميتها
    الجماعية عبر معلومات التدرج (Gradient-based Importance)، ثم إزالة
    المجموعات الأقل أهمية بالكامل - يليها إعادة ضبط سريع (LoRA)
    لاستعادة الأداء المفقود.
    """
    def __init__(self, model, importance_calibration_data):
        self.model = model
        self.calibration_data = importance_calibration_data

    def compute_head_importance(self, layer_idx: int) -> torch.Tensor:
        """
        تقدير أهمية كل رأس انتباه عبر حساسية الخسارة لإزالته
        (Taylor Expansion تقريبي: |gradient × weight|، مجمّعة على
        مستوى الرأس الكامل بدل الوزن الفردي).
        """
        self.model.zero_grad()
        for batch in self.calibration_data:
            outputs = self.model(**batch, labels=batch["input_ids"])
            outputs.loss.backward()

        attention_layer = self.model.get_attention_layer(layer_idx)
        n_heads = attention_layer.n_heads
        head_dim = attention_layer.head_dim

        head_importance_scores = []
        for head_idx in range(n_heads):
            start = head_idx * head_dim
            end = start + head_dim
            # الأهمية = مجموع |وزن × تدرجه| لكل أوزان هذا الرأس تحديدًا
            weight_slice = attention_layer.weight.data[start:end, :]
            grad_slice = attention_layer.weight.grad[start:end, :]
            importance = (weight_slice * grad_slice).abs().sum()
            head_importance_scores.append(importance.item())

        return torch.tensor(head_importance_scores)

    def prune_least_important_heads(self, layer_idx: int, n_heads_to_remove: int):
        """إزالة الرؤوس الأقل أهمية بالكامل (تقليم بنيوي حقيقي)."""
        importance = self.compute_head_importance(layer_idx)
        heads_to_remove = torch.topk(
            importance, n_heads_to_remove, largest=False
        ).indices.tolist()

        attention_layer = self.model.get_attention_layer(layer_idx)
        attention_layer.remove_heads(heads_to_remove)  # دالة مفاهيمية
        return self.model

    def post_pruning_recovery(self, recovery_dataset, lora_rank: int = 16):
        """
        خطوة ضرورية دائمًا بعد التقليم البنيوي: إعادة ضبط سريع
        وخفيف التكلفة (عبر LoRA - الجزء السابع) لاستعادة أكبر قدر
        ممكن من الأداء المفقود جراء إزالة المكوّنات البنيوية.
        """
        from peft import LoraConfig, get_peft_model
        lora_config = LoraConfig(
            r=lora_rank, lora_alpha=lora_rank * 2,
            target_modules=["q_proj", "v_proj"], lora_dropout=0.05,
        )
        recoverable_model = get_peft_model(self.model, lora_config)
        # تدريب LoRA قصير (عادة 1-2% فقط من حجم بيانات CPT الأصلية كافٍ)
        return recoverable_model


> 💡 نصيحة عملية: التقليم البنيوي دائمًا يتطلب مرحلة تعافي (Recovery Fine-tuning) بعده — عادة بضع مئات إلى آلاف خطوات تدريب بـ LoRA كافية لاستعادة 90%+ من الأداء المفقود، بتكلفة ضئيلة جدًا مقارنة بإعادة التدريب الكامل. تجاهل هذه الخطوة شبه مضمون أن يُنتج نموذجًا مُتدهورًا بشكل ملحوظ.


## 4. مثال محلول خطوة بخطوة

لنقارن Magnitude وWanda على مصفوفة أوزان صغيرة (3 صفوف × 4 أعمدة)، مع افتراض
أن تنشيطات الأعمدة الأربعة لها المعايير: `[2.0, 0.5, 1.5, 0.1]`.


In [ ]:
import torch

W = torch.tensor([
    [0.10,  0.90, -0.20, 0.05],
    [0.70, -0.30,  0.40, 0.02],
    [-0.60, 0.20,  0.15, 0.80],
])
act_norms = torch.tensor([2.0, 0.5, 1.5, 0.1])

# --- Magnitude: نحتفظ بأكبر قيمتين مطلقًا في كل صف ---
def magnitude_prune(W, keep=2):
    out = torch.zeros_like(W)
    for r in range(W.shape[0]):
        idx = W[r].abs().topk(keep).indices
        out[r, idx] = W[r, idx]
    return out

# --- Wanda: |W| × ||X|| ---
def wanda_prune(W, act_norms, keep=2):
    importance = W.abs() * act_norms.unsqueeze(0)
    out = torch.zeros_like(W)
    for r in range(W.shape[0]):
        idx = importance[r].topk(keep).indices
        out[r, idx] = W[r, idx]
    return out

print("Magnitude:\n", magnitude_prune(W))
print("Wanda:\n", wanda_prune(W, act_norms))


النتيجة الأهم: في الصف الثالث، يحتفظ Magnitude بالوزن `0.80` لأنه كبير، بينما
يطرده Wanda لأن العمود الرابع تنشيطه ضعيف جدًا (`0.1`). والعكس في العمود الأول:
Wanda يحتفظ بـ `-0.60` ويُسقط `0.20` لأن تنشيط العمود الأول قوي (`2.0`).
هذا يوضّح لماذا يكون Wanda أفضل عند نسب تقليم عالية: **الأهمية ليست حجم الوزن
وحده، بل حجم الوزن × حجم المدخل الذي يضربه فعليًا**.


## 5. نصائح وحيل

- **استخدم بيانات معايرة تمثيلية** لتوزيع استخدام النموذج الحقيقي، لا نصوصًا عشوائية.
- **ابدأ بـ Wanda** كخط أساس سريع، ثم جرّب SparseGPT إذا احتجت جودة أعلى عند نسب تقليم كبيرة.
- **فضّل التقليم البنيوي** عندما يكون هدفك تسريعًا فعليًا على عتاد عام.
- **خصّص مرحلة تعافٍ** (LoRA أو SFT قصير) بعد أي تقليم بنيوي؛ فهي ليست اختيارية.
- **قلّل نسبة التقليم في الطبقات الأولى والأخيرة**؛ فهي عادة أكثر حساسية.
- **استهدف نمط 2:4** إذا كان العتاد يدعم Sparse Tensor Cores، لتحصل على تسريع فعلي مع تقليم غير بنيوي.
- **قِس أكثر من Perplexity**: اختبر الاستدلال، والسلامة، والمهام التوليدية، لأن التقليم قد يضرّ قدرة دون أخرى.
- **لا تُطبّق تقليمين متتاليين دون تعافٍ بينهما**؛ فالخطأ يتراكم.


## 6. أخطاء شائعة ومصائد

تطبيق نسبة تقليم موحّدة (Uniform Sparsity) عبر كل طبقات النموذج — الأبحاث تُظهر أن الطبقات الأولى والأخيرة عادة أكثر حساسية للتقليم من الطبقات الوسطى؛ التقليم غير المتجانس (Non-uniform، بنسب أقل للطبقات الحساسة) غالبًا يُعطي نتائج أفضل.
تقليم نموذج ثم نشره مباشرة في الإنتاج دون قياس التدهور على معايير متنوعة (ليس فقط Perplexity) — التقليم قد يُضعِف قدرات محددة (كالاستدلال المنطقي) أكثر من غيرها بشكل غير متناسب.
الخلط بين "التقليم غير البنيوي بنسبة عالية" و"تسريع فعلي" — كما ذُكر، الأول يوفر الذاكرة نظريًا فقط ما لم يوجد دعم هاردوير مخصص (مثل أنماط 2:4 على NVIDIA Ampere+).


## 7. جدول مقارنة

| المعيار | التقليم غير البنيوي (Unstructured) | التقليم البنيوي (Structured) |
|---|---|---|
| ماذا يُزال | أوزان فردية متناثرة في أي مكان | مكوّنات كاملة (رؤوس انتباه، قنوات FFN، طبقات) |
| النمط الناتج | مصفوفة متناثرة (Sparse) غير منتظمة | مصفوفات أصغر لكن كثيفة (Dense) ومنتظمة |
| التسريع الفعلي على GPU عادي | ضعيف جدًا (يحتاج هاردوير/مكتبات متخصصة لاستغلال الندرة) | فعلي ومباشر (مصفوفات أصغر = حوسبة أقل تلقائيًا) |
| أقصى نسبة تقليم ممكنة مع أداء مقبول | عالية جدًا (حتى 90%+) | أقل (عادة 20-50%) |
| التعقيد التنفيذي | أبسط للحساب لكن أصعب للاستفادة الفعلية | أبسط للاستفادة الفعلية |

> ⚠️ خطأ شائع جوهري: الاعتقاد بأن أي تقليم (Pruning) يُسرِّع الاستدلال تلقائيًا. هذا غير صحيح للتقليم غير البنيوي على معظم أجهزة GPU التقليدية! بدون دعم هاردوير خاص (مثل NVIDIA Sparse Tensor Cores التي تدعم نمط 2:4 محدد)، فإن مصفوفة متناثرة بنسبة 90% لا تزال تُخزَّن وتُعالَج غالبًا كمصفوفة كثيفة عاديًا من ناحية عمليات الضرب المصفوفي القياسية، فتُفقَد فائدة السرعة رغم توفير الذاكرة النظري.


## 8. ملخص أفضل الممارسات

✅ استخدم Wanda كخط أساس سريع وفعّال قبل اللجوء لـ SparseGPT الأكثر تعقيدًا وتكلفة حسابية.
✅ للحصول على تسريع فعلي حقيقي على أي هاردوير، التزم بالتقليم البنيوي رغم أن أقصى نسبة تقليم ممكنة أقل.
✅ خصّص دائمًا مرحلة تعافي (LoRA recovery fine-tuning) بعد أي تقليم بنيوي.
✅ استخدم بيانات معايرة (Calibration Data) تمثيلية فعلًا لتوزيع استخدام النموذج المستهدف، وليس بيانات عشوائية.


## 9. أسئلة مقابلات

**سؤال: لماذا لا يُسرِّع التقليم غير البنيوي (Unstructured Pruning) الاستدلال فعليًا على معظم أجهزة GPU، رغم تقليل عدد الأوزان غير الصفرية بشكل كبير؟**

**إجابة نموذجية:** عمليات الضرب المصفوفي الكثيف (Dense Matrix Multiplication) على GPU مُحسَّنة بشدة عبر مكتبات مثل cuBLAS لاستغلال التوازي المصفوفي الكامل بافتراض أن كل العناصر (بما فيها الأصفار) ستُعالَج بنفس الطريقة المنتظمة. التقليم غير البنيوي يُنتج نمط تصفير "عشوائي" غير منتظم داخل المصفوفة، ولا يمكن لهذه المكتبات استغلال هذا التناثر لتقليل العمليات الفعلية ما لم يوجد دعم هاردوير/برمجيات مخصص (مثل Sparse Tensor Cores في NVIDIA Ampere، التي تدعم فقط نمط تناثر مُقيَّد جدًا وهو 2 من كل 4 عناصر = صفر). لذا، فإن مصفوفة متناثرة بنسبة 90% لا تزال، في غياب هذا الدعم المتخصص، تُخزَّن وتُعالَج كمصفوفة كثيفة تقريبًا من ناحية زمن التنفيذ الفعلي، رغم أن حجم تخزينها على القرص (إذا استُخدمت صيغة تخزين متناثرة مخصصة) قد يكون أصغر.


## 10. تمرين عملي

طبّق wanda_pruning على طبقات FFN في نموذج GPT-2 الصغير بنسب تقليم مختلفة (30%, 50%, 70%)، وقِس التدهور في Perplexity (الفصل 41) على نص اختباري عند كل نسبة. ثم طبّق مرحلة تعافٍ قصيرة بـ LoRA وقارن مدى استعادة الأداء.


## 11. مراجع للتعمّق

- Frantar & Alistarh (2023): SparseGPT: Massive Language Models Can Be Accurately Pruned in One-Shot
- Sun et al. (2023): A Simple and Effective Pruning Approach for Large Language Models (Wanda)
- Ma et al. (2023): LLM-Pruner: On the Structural Pruning of Large Language Models
- Han et al. (2015): Learning both Weights and Connections for Efficient Neural Networks


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — إعادة استخدام النماذج / الفصل 27: التقليم وضغط النماذج: Pruning.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
